# Extração de Dados


Todos os dados brutos foram baixados manualmente em formato CSV de https://dados.turismo.gov.br/dataset/chegada-de-turistas-internacionais


### Atualização dos Dados

1. Baixe o novo CSV a partir do link acima e salve-o em `data/raw/`. Qualquer nome de arquivo
   terminado em `.csv` é lido.
2. O ministério publica o ano corrente como arquivos **acumulados** (por exemplo
   `base_chegadas_2026_acum_08_agosto.csv`, janeiro–agosto). Quando um arquivo acumulado mais recente
   do mesmo ano chegar, **delete o mais antigo**. Cada ano deve vir de exatamente um
   arquivo; caso contrário, seus meses seriam contados duas vezes. A extração abaixo é interrompida com um
   erro se dois arquivos contiverem o mesmo ano.
3. Execute novamente os notebooks em ordem. A divisão treino/teste (notebook 2) sempre separa
   os **últimos 12 meses** dos dados, e todos os notebooks posteriores leem suas datas a partir da
   divisão, de modo que nenhuma data precisa ser editada manualmente.

Se um novo arquivo utilizar um nome de coluna que não esteja em `COLUMN_MAPPING`, a extração é interrompida e
lista as colunas do arquivo. Adicione o novo nome a `COLUMN_MAPPING` e execute novamente.

In [1]:
import os

import pandas as pd
from tqdm.auto import tqdm

ROOT_PATH = os.path.dirname(os.getcwd())
DATA_FOLDER_PATH = os.path.join(ROOT_PATH, 'data')

## Importação de Dados


Entendendo como os nomes das colunas mudam ao longo do tempo


In [2]:
last_adopted_column_pattern = ''

for file in sorted(os.listdir(os.path.join(DATA_FOLDER_PATH, 'raw'))):
    if not file.endswith('.csv'):
        continue

    df = pd.read_csv(
        os.path.join(DATA_FOLDER_PATH, 'raw', file),
        encoding='latin-1',
        low_memory=False,
        sep=';',
    )

    if last_adopted_column_pattern != str(df.columns):
        last_adopted_column_pattern = str(df.columns)

        print(f'Arquivo: {file}')
        print(last_adopted_column_pattern)
        print('\n')


Arquivo: base_chegadas_2026_acum_08_agosto.csv
Index(['Via_de_acesso', 'UF', 'nome_pais_correto', 'mes', 'ano', 'Chegadas'], dtype='object')


Arquivo: chegadas_1989.csv
Index(['Continente', 'Ordem continente', 'País', 'Ordem país', 'UF',
       'Ordem UF', 'Via de acesso', 'Ordem via de acesso', 'ano', 'Mês',
       'Ordem mês', 'Chegadas'],
      dtype='object')


Arquivo: chegadas_2000.csv
Index(['Continente', 'Ordem continente', 'País', 'Ordem país', 'UF',
       'Ordem UF', 'Via de acesso', 'Ordem via de acesso', 'Ano', 'Mês',
       'Ordem mês', 'Chegadas'],
      dtype='object')


Arquivo: chegadas_2015.csv
Index(['Continente', 'Ordem continente', 'País', 'Ordem país', 'UF',
       'Ordem UF', 'Via de acesso', 'Ordem via de acesso', 'ano', 'Mês',
       'Ordem mês', 'Chegadas'],
      dtype='object')


Arquivo: chegadas_2016.csv
Index(['Continente', 'cod continente', 'País', 'cod pais', 'UF', 'cod uf',
       'Via', 'cod via', 'ano', 'Mês', 'cod mes', 'Chegadas'],
      dtype='o

In [3]:
COLUMN_MAPPING = {
    'access_route': ['Via de acesso', 'Via', 'Via_de_acesso'],
    'state': ['UF'],
    'origin_country': ['País', 'nome_pais_correto'],
    'month': ['Mês', 'mes'],
    'year': ['ano', 'Ano'],
    'arrival_count': ['Chegadas'],
}

SELECTED_COLUMNS = list(COLUMN_MAPPING.keys())


def standardize_columns(df: pd.DataFrame, file_name: str) -> pd.DataFrame:
    rename_map = {
        variant: target
        for target, variants in COLUMN_MAPPING.items()
        for variant in variants
        if variant in df.columns
    }
    df = df.rename(columns=rename_map)
    missing = [column for column in SELECTED_COLUMNS if column not in df.columns]
    if missing:
        raise KeyError(
            f'{file_name}: nenhuma coluna de origem para {missing}. '
            f'Colunas no arquivo: {list(df.columns)}. '
            'Adicione o novo nome da coluna ao COLUMN_MAPPING.'
        )
    return df[SELECTED_COLUMNS]


Unificando os dados


In [4]:
dfs = []
files_by_year = {}

csv_files = sorted(
    file
    for file in os.listdir(os.path.join(DATA_FOLDER_PATH, 'raw'))
    if file.endswith('.csv')
)

for arrival_file in tqdm(csv_files, desc='Lendo arquivos de chegadas'):
    try:
        df = pd.read_csv(
            os.path.join(DATA_FOLDER_PATH, 'raw', arrival_file),
            encoding='utf-8',
            low_memory=False,
            sep=';',
        )

    except UnicodeDecodeError:
        df = pd.read_csv(
            os.path.join(DATA_FOLDER_PATH, 'raw', arrival_file),
            encoding='latin-1',
            low_memory=False,
            sep=';',
        )

    df = standardize_columns(df, arrival_file)
    for year in df['year'].dropna().astype(int).unique():
        files_by_year.setdefault(int(year), []).append(arrival_file)
    dfs.append(df)

# cada ano deve vir de exatamente um arquivo (versões acumuladas do mesmo ano
# contariam os meses sobrepostos duas vezes)
duplicated_years = {
    year: files for year, files in files_by_year.items() if len(files) > 1
}
if duplicated_years:
    raise ValueError(
        'O mesmo ano aparece em mais de um CSV. Mantenha apenas o arquivo mais recente '
        f'de cada ano em data/raw: {duplicated_years}'
    )


df = pd.concat(dfs).reset_index(drop=True)

df.head()


Lendo arquivos de chegadas:   0%|          | 0/38 [00:00<?, ?it/s]

,access_route,state,origin_country,month,year,arrival_count
0,Aérea,Bahia,Afeganistão,Janeiro,2026,1.0
1,Aérea,São Paulo,Afeganistão,Janeiro,2026,4.0
2,Terrestre,Paraná,Afeganistão,Janeiro,2026,1.0
3,Terrestre,Rio Grande do Sul,Afeganistão,Janeiro,2026,1.0
4,Marítima,Rio de Janeiro,Afeganistão,Janeiro,2026,3.0


Exportando


In [5]:
df.to_parquet(
    os.path.join(DATA_FOLDER_PATH, 'raw', 'raw_arrivals_ptbr.parquet'), index=False
)
